# 01 — Exploratory Data Analysis (EDA)

## What to do
Load and understand the dataset structure, distributions, relationships, and quality issues.

## What you'll learn
- How to inspect raw data
- Identify missing values and data types
- Understand target variable distribution
- Spot relationships between features and target
- Detect potential issues before modeling

## Step 1: Setup and imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (12, 6)

print('Libraries loaded successfully!')

## Step 2: Load the dataset

**Action:** Load the CSV file from the data folder.

**Insight:** This shows the dataset exists and is readable. Check the file path if you get an error.

In [ ]:
df = pd.read_csv('data/diabetes_prediction_dataset.csv')

print(f'Dataset loaded successfully!')
print(f'Shape: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'\nFirst 5 rows:')
display(df.head())

## Step 3: Inspect data structure

**Action:** Check column names, data types, and get dataset info.

**Insight:** You'll see what variables you have and whether they are numeric, categorical, or other types. This guides your preprocessing strategy.

In [ ]:
print('Column names and types:')
print(df.info())

print('\n' + '='*50)
print('Data types summary:')
print(df.dtypes)

## Step 4: Check for missing values

**Action:** Count null values in each column.

**Insight:** Missing values must be handled before modeling. A high percentage means we need a strategy (fill, drop, or predict).

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': missing.values,
    'Missing_Percentage': missing_pct.values
}).sort_values('Missing_Percentage', ascending=False)

print('Missing values analysis:')
display(missing_df[missing_df['Missing_Count'] > 0])

if missing.sum() == 0:
    print('✓ No missing values found! Data is clean.')

## Step 5: Check for duplicates

**Action:** Count duplicate rows.

**Insight:** Duplicates can skew model training. If found, they should be removed.

In [ ]:
duplicates = df.duplicated().sum()
print(f'Number of duplicate rows: {duplicates}')

if duplicates > 0:
    print(f'Duplicates found! {duplicates} rows need to be removed.')
else:
    print('✓ No duplicates found!')

## Step 6: Summary statistics

**Action:** Display descriptive statistics for all columns.

**Insight:** You'll see the range, mean, and quartiles of your data. This helps spot outliers and understand feature scales.

In [ ]:
print('Descriptive statistics:')
display(df.describe().T)

## Step 7: Analyze the target variable

**Action:** Check the distribution of the diabetes target.

**Insight:** Class imbalance affects model evaluation. If one class dominates, you need balanced metrics (F1, AUC) instead of accuracy.

In [ ]:
target_col = 'diabetes'  # Adjust if your target column has a different name

print('Target variable distribution:')
target_counts = df[target_col].value_counts()
target_pct = (target_counts / len(df)) * 100

target_df = pd.DataFrame({
    'Class': target_counts.index,
    'Count': target_counts.values,
    'Percentage': target_pct.values
})
display(target_df)

# Visualize target distribution
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x=target_col, palette='Set2')
plt.title('Target Variable Distribution')
plt.xlabel('Diabetes Status')
plt.ylabel('Count')
plt.show()

# Calculate class balance
class_ratio = target_counts.iloc[0] / target_counts.iloc[1]
print(f'\nClass ratio (majority/minority): {class_ratio:.2f}:1')
if class_ratio > 3:
    print('⚠ Data is imbalanced! Use balanced metrics for evaluation.')

## Step 8: Numeric features distribution

**Action:** Visualize histograms and boxplots for numeric features.

**Insight:** Skewed distributions may need transformation. Outliers visible in boxplots need attention.

In [ ]:
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()

# Remove target from visualization if it's numeric
if target_col in numeric_cols:
    numeric_cols.remove(target_col)

# Plot distributions
n_cols = len(numeric_cols)
fig, axes = plt.subplots(n_cols, 2, figsize=(15, 4*n_cols))
if n_cols == 1:
    axes = [axes]

for i, col in enumerate(numeric_cols):
    # Histogram
    axes[i][0].hist(df[col], bins=30, color='steelblue', edgecolor='black')
    axes[i][0].set_title(f'{col} - Distribution')
    axes[i][0].set_xlabel(col)
    axes[i][0].set_ylabel('Frequency')
    
    # Boxplot
    axes[i][1].boxplot(df[col])
    axes[i][1].set_title(f'{col} - Boxplot (Outliers)')
    axes[i][1].set_ylabel(col)

plt.tight_layout()
plt.show()

## Step 9: Categorical features distribution

**Action:** Count unique values and visualize categorical features.

**Insight:** Rare categories may need grouping. High cardinality needs encoding strategy.

In [ ]:
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print('Categorical features:')
for col in categorical_cols:
    print(f'\n{col}:')
    print(df[col].value_counts())

# Visualize categorical features
if categorical_cols:
    n_cols = len(categorical_cols)
    fig, axes = plt.subplots(1, n_cols, figsize=(5*n_cols, 4))
    if n_cols == 1:
        axes = [axes]
    
    for i, col in enumerate(categorical_cols):
        df[col].value_counts().plot(kind='bar', ax=axes[i], color='coral')
        axes[i].set_title(f'{col} Distribution')
        axes[i].set_ylabel('Count')
    
    plt.tight_layout()
    plt.show()

## Step 10: Correlation analysis

**Action:** Calculate and visualize correlations between numeric features and target.

**Insight:** High correlations indicate strong predictors. Multicollinearity between features can affect some models.

In [ ]:
# Correlation with target
numeric_with_target = df.select_dtypes(include=['number']).columns.tolist()

correlation_with_target = df[numeric_with_target].corr()[target_col].sort_values(ascending=False)
print('Correlation with target (sorted):')
display(correlation_with_target)

# Full correlation heatmap
plt.figure(figsize=(10, 8))
correlation_matrix = df[numeric_with_target].corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap (Numeric Features)')
plt.tight_layout()
plt.show()

## Step 11: Relationships between features and target

**Action:** Visualize how key features differ by target class.

**Insight:** Clear separation between classes indicates good predictive power.

In [ ]:
# Select top correlated numeric features
top_features = correlation_with_target.drop(target_col).head(5).index.tolist()

fig, axes = plt.subplots(1, len(top_features), figsize=(5*len(top_features), 4))
if len(top_features) == 1:
    axes = [axes]

for i, feature in enumerate(top_features):
    sns.boxplot(data=df, x=target_col, y=feature, ax=axes[i], palette='Set2')
    axes[i].set_title(f'{feature} by {target_col}')

plt.tight_layout()
plt.show()

## Summary of EDA Findings

Complete the EDA summary below:

In [ ]:
print('='*60)
print('EDA SUMMARY')
print('='*60)
print(f'Total rows: {len(df):,}')
print(f'Total columns: {len(df.columns)}')
print(f'\nNumeric features: {len(numeric_cols)}')
print(f'Categorical features: {len(categorical_cols)}')
print(f'\nMissing values: {df.isnull().sum().sum()}')
print(f'Duplicate rows: {df.duplicated().sum()}')
print(f'\nTarget variable imbalance: {"Yes - needs handling" if class_ratio > 2 else "No - balanced"}')
print(f'\nTop 3 correlated features with target:')
for i, (feat, corr) in enumerate(correlation_with_target.drop(target_col).head(3).items(), 1):
    print(f'  {i}. {feat}: {corr:.3f}')
print('='*60)
print('\n✓ EDA Complete! Ready for data cleaning.')